# 05 · Pre-test planning (sample size, power, MDE) — `foe.pretest`

**Hexkit surfaces:** `pages/pre_test_analysis.py` (`_sample_size_per_variant`, `_power_two_sample`,
`_relative_mde`, `get_z_alpha`, `compound_per_visitor_moments`), `utility/automation_engine.py`
(`calculate_mde_table`, `calculate_mde_from_forecast`).

| FOE function | R reference |
|---|---|
| `calculate_fixed_sample_size` (binomial, pooled-null form), incl. one-sided and Holm-Bonferroni for 3+ arms | `stats::power.prop.test` — same formula, solved numerically |
| `_power_two_sample` with pooled binomial variance | `power.prop.test(strict = TRUE)` |
| `_sample_size_per_variant` (continuous, equal variance) | `TrialSize::TwoSampleMean.Equality` |
| `_power_two_sample` (continuous, equal variance) | `pwr::pwr.norm.test` with d = δ / √(2σ²) |
| `calculate_mde_table`, `calculate_mde_from_forecast` (`_relative_mde`) | `pwr::pwr.norm.test` power function, inverted for d |
| `compound_per_visitor_moments` | moments of a large simulated zero-inflated Gamma sample in R |

Not covered: the CV-constant (unequal-variance) branch has no off-the-shelf R equivalent; its
formula differs from the equal-variance one only in the H1 variance term, which is exercised
through `_power_two_sample` below.

In [1]:
import os
os.environ.setdefault("RPY2_CFFI_MODE", "ABI")  # use rpy2 ABI mode; avoids a noisy API-mode probe
%load_ext rpy2.ipython

import numpy as np
import pandas as pd
from scipy.stats import norm

from foe.core.models import AlternativeHypothesis as Alt
from foe.pretest.operations import PretestEngine as PE
from validation_helpers import Checks

checks = Checks()

## 1. Binomial sample size vs `power.prop.test`

In [2]:
grid = []
for p in [0.01, 0.03, 0.10, 0.35]:
    for mde in [0.05, 0.10, 0.20]:
        for alt, k in [("two-sided", 2), ("greater", 2), ("two-sided", 4)]:
            out = PE.calculate_fixed_sample_size(p, mde, num_variants=k, alpha=0.05, power=0.8,
                                                 alternative=Alt(alt))
            grid.append(dict(p=p, mde=mde, alternative=alt, k=k, n_py=out["n_per_variant"]))
grid = pd.DataFrame(grid)
grid.head()

,p,mde,alternative,k,n_py
0,0.01,0.05,two-sided,2,637010
1,0.01,0.05,greater,2,501773
2,0.01,0.05,two-sided,4,849667
3,0.01,0.10,two-sided,2,163095
4,0.01,0.10,greater,2,128470


In [3]:
%%R -i grid -o n_r
n_r <- sapply(seq_len(nrow(grid)), function(i) {
  g <- grid[i, ]
  # FOE: 3+ arms use the most conservative Holm step, alpha / (k - 1).
  a <- 0.05 / max(g$k - 1, 1)
  power.prop.test(p1 = g$p, p2 = g$p * (1 + g$mde), sig.level = a, power = 0.8,
                  alternative = if (g$alternative == "two-sided") "two.sided" else "one.sided",
                  tol = 1e-12)$n
})

In [4]:
for i, g in grid.iterrows():
    # power.prop.test returns the unrounded n; FOE rounds up.
    checks.add(f"binomial n  p={g.p} mde={g.mde} {g.alternative} k={g.k}",
               g.n_py, np.ceil(n_r[i] - 1e-9), atol=0)

## 2. Power (two-sample z) with pooled binomial variance vs `power.prop.test(strict = TRUE)`

In [5]:
pw = []
for p1, p2, n in [(0.03, 0.033, 20_000), (0.10, 0.11, 8_000), (0.25, 0.27, 3_000), (0.05, 0.06, 6_000)]:
    pbar = (p1 + p2) / 2
    for alt in ["two-sided", "greater"]:
        z = PE.get_z_alpha(2, 0.05, Alt(alt))
        pw.append(dict(p1=p1, p2=p2, n=n, alternative=alt, power_py=PE._power_two_sample(
            delta=p2 - p1, var_null_sum=2 * pbar * (1 - pbar),
            var_alt_sum=p1 * (1 - p1) + p2 * (1 - p2), n_per_variant=n, z_alpha=z,
            alternative=Alt(alt))))
pw = pd.DataFrame(pw)

In [6]:
%%R -i pw -o pw_r
pw_r <- sapply(seq_len(nrow(pw)), function(i) {
  g <- pw[i, ]
  power.prop.test(n = g$n, p1 = g$p1, p2 = g$p2, sig.level = 0.05, strict = TRUE,
                  alternative = if (g$alternative == "two-sided") "two.sided" else "one.sided")$power
})

In [7]:
for i, g in pw.iterrows():
    checks.add(f"binomial power p1={g.p1} p2={g.p2} n={g.n} {g.alternative}", g.power_py, pw_r[i], rtol=1e-10)

## 3. Continuous KPI: sample size and power (equal variance)

In [8]:
cont = []
for mean, sd in [(55.0, 70.0), (2.4, 3.1), (120.0, 95.0)]:
    v = sd ** 2
    for mde in [0.03, 0.05, 0.10]:
        delta = mean * mde
        z_a, z_b = PE.get_z_alpha(2, 0.05, Alt.TWO_SIDED), norm.ppf(0.8)
        n = PE._sample_size_per_variant(delta, 2 * v, 2 * v, z_a, z_b)
        n_fixed = 5_000
        power = PE._power_two_sample(delta, 2 * v, 2 * v, n_fixed, z_a, Alt.TWO_SIDED)
        power_1s = PE._power_two_sample(delta, 2 * v, 2 * v, n_fixed,
                                        PE.get_z_alpha(2, 0.05, Alt.GREATER), Alt.GREATER)
        cont.append(dict(mean=mean, sd=sd, mde=mde, delta=delta, n_py=n, n_fixed=n_fixed,
                         power_py=power, power_1s_py=power_1s))
cont = pd.DataFrame(cont)

In [9]:
%%R -i cont -o cont_r
suppressPackageStartupMessages({library(TrialSize); library(pwr)})
cont_r <- do.call(rbind, lapply(seq_len(nrow(cont)), function(i) {
  g <- cont[i, ]
  d <- g$delta / sqrt(2 * g$sd^2)          # two-sample z == one-sample z on d / sqrt(2)
  data.frame(
    n = TwoSampleMean.Equality(alpha = 0.05, beta = 0.2, sigma = g$sd, k = 1, margin = g$delta),
    power = pwr.norm.test(d = d, n = g$n_fixed, sig.level = 0.05, alternative = "two.sided")$power,
    power_1s = pwr.norm.test(d = d, n = g$n_fixed, sig.level = 0.05, alternative = "greater")$power
  )
}))

In [10]:
for i, g in cont.iterrows():
    r = cont_r.iloc[i]
    tag = f"mean={g['mean']} sd={g.sd} mde={g.mde}"
    checks.add(f"continuous n          {tag}", g.n_py, np.ceil(r.n - 1e-9), atol=0)
    checks.add(f"continuous power 2s   {tag}", g.power_py, r.power, rtol=1e-10)
    checks.add(f"continuous power 1s   {tag}", g.power_1s_py, r.power_1s, rtol=1e-10)

## 4. MDE tables (static traffic and from a forecast)

FOE's MDE is `(z_α + z_β) · √(2σ²/n) / mean`, which ignores the far rejection tail. R solves
`pwr.norm.test`'s power function for d; for a two-sided test that tail contributes ~1e-6 to the power,
so the tolerance is 1e-4 relative. One-sided cases match to solver precision.

In [11]:
mde_cases = []
for weekly, conv, k, alt in [(70_000, 2_100, 2, "two-sided"), (70_000, 2_100, 3, "two-sided"),
                             (12_000, 1_500, 2, "greater")]:
    tbl = PE.calculate_mde_table(k, weekly, conv, risk_pct=95, trust_pct=80, alternative=Alt(alt))["table"]
    for row in tbl:
        mde_cases.append(dict(source="static", k=k, alternative=alt, mean=conv / weekly,
                              var=conv / weekly * (1 - conv / weekly),
                              n=row["Visitors_Per_Variant"], mde_py=row["MDE"]))

# Seasonal path: 42 days of forecast records (pred_count = visitors, pred_value = revenue).
rng = np.random.default_rng(5)
days = pd.date_range("2026-01-05", periods=42)
counts = 9_000 + 2_000 * np.sin(np.arange(42) / 7 * 2 * np.pi) + rng.normal(0, 300, 42)
records = [{"ds": d, "pred_count": float(c), "pred_value": float(c * 1.9)} for d, c in zip(days, counts)]
cv = 6.5
tbl = PE.calculate_mde_from_forecast(records, num_variants=2, risk_pct=95, trust_pct=80, kpi_cv=cv)["table"]
for week, row in enumerate(tbl, start=1):
    n = sum(r["pred_count"] for r in records[: week * 7]) / 2
    mde_cases.append(dict(source="forecast", k=2, alternative="two-sided", mean=row["Mean"],
                          var=(cv * row["Mean"]) ** 2, n=n, mde_py=row["MDE"]))
mde_cases = pd.DataFrame(mde_cases)
mde_cases.head(8)

,source,k,alternative,mean,var,n,mde_py
0,static,2,two-sided,0.03,0.0291,35000.0,12.042317
1,static,2,two-sided,0.03,0.0291,70000.0,8.515204
2,static,2,two-sided,0.03,0.0291,105000.0,6.952635
3,static,2,two-sided,0.03,0.0291,140000.0,6.021158
4,static,2,two-sided,0.03,0.0291,175000.0,5.385488
5,static,2,two-sided,0.03,0.0291,210000.0,4.916255
6,static,3,two-sided,0.03,0.0291,23334.0,16.230150
7,static,3,two-sided,0.03,0.0291,46668.0,11.476449


In [12]:
%%R -i mde_cases -o mde_r
suppressPackageStartupMessages(library(pwr))
mde_r <- sapply(seq_len(nrow(mde_cases)), function(i) {
  g <- mde_cases[i, ]
  a <- 0.05 / max(g$k - 1, 1)
  alt <- if (g$alternative == "two-sided") "two.sided" else "greater"
  # pwr.norm.test can solve for d itself, but its internal uniroot tolerance (~1e-4) is too
  # loose for this comparison, so invert its power function with a tight tolerance instead.
  d <- uniroot(function(d) pwr.norm.test(d = d, n = g$n, sig.level = a, alternative = alt)$power - 0.8,
               c(1e-8, 10), tol = 1e-14)$root
  d * sqrt(2 * g$var) / g$mean * 100     # relative MDE in %
})

In [13]:
for i, g in mde_cases.iterrows():
    checks.add(f"MDE % {g.source} k={g.k} {g.alternative} n={g.n:.0f}", g.mde_py, mde_r[i], rtol=1e-4)

## 5. Revenue-per-visitor moments (`compound_per_visitor_moments`)

In [14]:
cr, k_shape, theta = 0.035, 1.9, 38.0
txn_mean, txn_var = k_shape * theta, k_shape * theta ** 2
rpv_mean, rpv_var = PE.compound_per_visitor_moments(txn_mean, txn_var, cr)
rpv_mean, rpv_var

(2.527, 272.08967100000007)

In [15]:
%%R -i cr -i k_shape -i theta -o sim_r
set.seed(1)
n <- 4e6
x <- ifelse(runif(n) < cr, rgamma(n, shape = k_shape, scale = theta), 0)
sim_r <- c(mean(x), var(x))
sim_r

[1]

   2.542374

 273.928146

In [16]:
checks.add("RPV mean (vs simulation)", rpv_mean, sim_r[0], rtol=0.01)
checks.add("RPV variance (vs simulation)", rpv_var, sim_r[1], rtol=0.01)

True

## Summary

In [17]:
checks.report()

97 checks, 97 passed, 0 failed

Largest differences:
                               check  hexkit       R    abs_diff               tolerance
        RPV variance (vs simulation)  272.09 273.928     1.83847   atol=1e-08, rtol=0.01
            RPV mean (vs simulation)   2.527 2.54237   0.0153742   atol=1e-08, rtol=0.01
MDE % forecast k=2 two-sided n=31311 14.5541 14.5541 1.78244e-05 atol=1e-08, rtol=0.0001
  MDE % static k=2 two-sided n=35000 12.0423 12.0423 1.47482e-05 atol=1e-08, rtol=0.0001
MDE % forecast k=2 two-sided n=63003 10.2601 10.2601 1.25656e-05 atol=1e-08, rtol=0.0001


,check,hexkit,R,abs_diff,tolerance,pass
0,binomial n p=0.01 mde=0.05 two-sided k=2,637010.000000,637010.000000,0.000000,"atol=0, rtol=0",True
1,binomial n p=0.01 mde=0.05 greater k=2,501773.000000,501773.000000,0.000000,"atol=0, rtol=0",True
2,binomial n p=0.01 mde=0.05 two-sided k=4,849667.000000,849667.000000,0.000000,"atol=0, rtol=0",True
3,binomial n p=0.01 mde=0.1 two-sided k=2,163095.000000,163095.000000,0.000000,"atol=0, rtol=0",True
4,binomial n p=0.01 mde=0.1 greater k=2,128470.000000,128470.000000,0.000000,"atol=0, rtol=0",True
...,...,...,...,...,...,...
92,MDE % forecast k=2 two-sided n=125198,7.278362,7.278353,0.000009,"atol=1e-08, rtol=0.0001",True
93,MDE % forecast k=2 two-sided n=156126,6.517704,6.517696,0.000008,"atol=1e-08, rtol=0.0001",True
94,MDE % forecast k=2 two-sided n=187350,5.949842,5.949835,0.000007,"atol=1e-08, rtol=0.0001",True
95,RPV mean (vs simulation),2.527000,2.542374,0.015374,"atol=1e-08, rtol=0.01",True
